# Dispensables — lo que sobra

Eliminarlos no cambia el comportamiento, pero mejora mucho la claridad.

## Serie: Refactorización y Code Smells

Este contenido está dividido en 6 notebooks — uno por categoría de code
smell (clasificación de refactoring.guru) más un cierre de ejercicios:

1. `01_bloaters.ipynb` — Bloaters
2. `02_object_orientation_abusers.ipynb` — Object-Orientation Abusers
3. `03_change_preventers.ipynb` — Change Preventers
4. **`04_dispensables.ipynb`** — Dispensables
5. `05_couplers.ipynb` — Couplers
6. `06_ejercicios_autoevaluacion.ipynb` — Ejercicios y autoevaluación


### 1. Duplicate Code (código duplicado) — PRIORIDAD #1

**Definición:** La misma estructura de código (idéntica o casi idéntica) aparece repetida en más de un lugar del proyecto.

**Síntoma:** "El código limpio no contiene duplicación": cada copia es un lugar más donde hay que recordar aplicar el mismo cambio.

**Técnica de refactor:** Extract Method

#### Con el smell

In [ ]:
def precio_regular(base):
    descuento = base * 0.05
    if base > 100:
        descuento += base * 0.02
    return base - descuento

def precio_vip(base):
    descuento = base * 0.05
    if base > 100:
        descuento += base * 0.02
    descuento += base * 0.03  # descuento extra VIP
    return base - descuento

print(precio_regular(150))
print(precio_vip(150))

#### Refactorizado

In [ ]:
def _descuento_base(base):
    descuento = base * 0.05
    if base > 100:
        descuento += base * 0.02
    return descuento

def precio_regular(base):
    return base - _descuento_base(base)

def precio_vip(base):
    descuento = _descuento_base(base) + base * 0.03
    return base - descuento

print(precio_regular(150))
print(precio_vip(150))

**Explicación:** `Extract Method` saca la lógica de descuento compartida a `_descuento_base`. Si mañana cambia la regla del descuento base, se edita una sola función y ambos precios quedan actualizados automáticamente.

### 2. Dead Code (código muerto)

**Definición:** Variables, métodos o clases que ya no se usan en ninguna parte del sistema.

**Síntoma:** Sobrevive a una migración o refactor anterior porque nadie se atrevió a borrarlo "por si acaso".

**Técnica de refactor:** Eliminar directamente (el control de versiones es el respaldo)

#### Con el smell

In [ ]:
def calcular_envio(peso):
    return peso * 1500

def calcular_envio_legacy(peso, zona):
    # tarifa antigua, ya no se usa desde que se
    # migró a calcular_envio(); nadie la llama
    if zona == "urbana":
        return peso * 1200
    return peso * 1800

total = calcular_envio(5)
print(total)

#### Refactorizado

In [ ]:
def calcular_envio(peso):
    return peso * 1500

# calcular_envio_legacy() se eliminó: no había
# ninguna llamada en el código ni en las pruebas.
# El historial de git conserva la versión anterior
# si en algún momento hace falta consultarla.

total = calcular_envio(5)
print(total)

**Explicación:** El código muerto se elimina directamente, sin comentarlo ni "guardarlo por si acaso": el sistema de control de versiones ya cumple esa función, y cada línea de código vivo que se mantiene es una línea que hay que seguir leyendo y entendiendo.

### 3. Speculative Generality (generalidad especulativa)

**Definición:** Abstracciones (clases abstractas, hooks, parámetros) creadas anticipadamente "por si se necesitan en el futuro", que nunca llegan a usarse.

**Síntoma:** Solo existe una subclase real, pero la jerarquía está preparada para muchas más que jamás aparecieron.

**Técnica de refactor:** Collapse Hierarchy

#### Con el smell

In [ ]:
class ProcesadorAbstracto:
    def preprocesar(self, datos):
        return datos  # hook nunca usado

    def procesar(self, datos):
        raise NotImplementedError

    def posprocesar(self, resultado):
        return resultado  # hook nunca usado

class ProcesadorCSV(ProcesadorAbstracto):
    def procesar(self, datos):
        return datos.strip().split(",")

p = ProcesadorCSV()
print(p.procesar("a, b, c"))

#### Refactorizado

In [ ]:
class ProcesadorCSV:
    def procesar(self, datos):
        return datos.strip().split(",")

# Se eliminó la superclase especulativa: nunca
# existió un segundo procesador ni se necesitaron
# los hooks pre/posprocesar().

p = ProcesadorCSV()
print(p.procesar("a, b, c"))

**Explicación:** `Collapse Hierarchy` fusiona la única subclase real con su superclase especulativa. Si en el futuro aparece un segundo procesador real, se extrae la abstracción común en ese momento — no antes.

### 4. Comments (comentarios)

**Definición:** Comentarios que existen para explicar un bloque de código confuso, en lugar de que el propio código sea claro por sí mismo.

**Síntoma:** Un comentario largo justo antes de una condición o bloque complicado: es una señal de que ese código necesita un nombre, no una explicación aparte.

**Técnica de refactor:** Extract Method / Rename Method

#### Con el smell

In [ ]:
class Cliente:
    def __init__(self, vip):
        self.vip = vip

class Pedido:
    def __init__(self, cliente, monto):
        self.cliente = cliente
        self.monto = monto

def procesar(pedido):
    # revisa si el cliente es VIP y si el monto supera 100
    # para aplicar un descuento adicional del 5%
    if pedido.cliente.vip and pedido.monto > 100:
        pedido.monto *= 0.95
    return pedido.monto

print(procesar(Pedido(Cliente(True), 150)))

#### Refactorizado

In [ ]:
class Cliente:
    def __init__(self, vip):
        self.vip = vip

class Pedido:
    def __init__(self, cliente, monto):
        self.cliente = cliente
        self.monto = monto

def _aplica_descuento_vip(pedido):
    return pedido.cliente.vip and pedido.monto > 100

def procesar(pedido):
    if _aplica_descuento_vip(pedido):
        pedido.monto *= 0.95
    return pedido.monto

print(procesar(Pedido(Cliente(True), 150)))

**Explicación:** `Extract Method` + `Rename Method` convierten la condición comentada en una función con un nombre autoexplicativo (`_aplica_descuento_vip`). El comentario deja de ser necesario porque el código ahora dice lo mismo que decía el comentario.

### 5. Lazy Class (clase perezosa)

**Definición:** Una clase que hace tan poco que no justifica el costo de mantenerla por separado.

**Síntoma:** Una clase con un solo campo y quizás un getter, usada por una única clase cliente, sin comportamiento propio relevante.

**Técnica de refactor:** Inline Class

#### Con el smell

In [ ]:
class Direccion:
    def __init__(self, calle):
        self.calle = calle

    def obtener_calle(self):
        return self.calle

class Cliente:
    def __init__(self, nombre, direccion):
        self.nombre = nombre
        self.direccion = direccion

cliente = Cliente("Ana", Direccion("Calle 10 # 5-20"))
print(cliente.direccion.obtener_calle())

#### Refactorizado

In [ ]:
class Cliente:
    def __init__(self, nombre, calle):
        self.nombre = nombre
        self.calle = calle  # ya no hace falta una clase aparte

cliente = Cliente("Ana", "Calle 10 # 5-20")
print(cliente.calle)

**Explicación:** `Inline Class` elimina `Direccion` y mueve su único campo directamente a `Cliente`, que era su única usuaria. Una clase menos que mantener, sin perder ninguna funcionalidad.